# Virtualize a real study: the Allen Institute Human Immune Health Atlas

This is the same workflow as the [intro notebook](virtualize_h5ad_collection.ipynb),
now on **real published data**: the Allen Institute for Immunology
[Human Immune Health Atlas](https://cellxgene.cziscience.com/collections/77f9d7e9-5675-49c3-abed-ce02f39eef1b)
(Nature 2025), distributed on CZ CELLxGENE as one `.h5ad` per cell type — all
sharing the same 32,357 genes.

**The point isn't saving disk space — it's access.** A downstream user opens the
whole study as one `AnnData` in a fraction of a second, then reads only the cells
they touch, straight from the original files. No concatenation step, no loading
gigabytes into memory, no second copy of the data to manage — and what they read
is bit-for-bit the original. The full atlas is 1.8M cells across 8 files; the
same call scales to all of them.

## Get two of the files

We use the two smallest cell-type files (~56 MB and ~331 MB). The download is a
single request each and is skipped if the files are already present — please
don't re-download needlessly.

In [1]:
import os, urllib.request

FILES = {
    'other_cells.h5ad': 'https://datasets.cellxgene.cziscience.com/70dd5115-cec8-49ad-a2fb-f985067b193f.h5ad',
    'dcs.h5ad':         'https://datasets.cellxgene.cziscience.com/4f925e42-3e42-4e2d-8394-4e88bf9a0ff2.h5ad',
}
os.makedirs('data/allen', exist_ok=True)
for name, url in FILES.items():
    dest = f'data/allen/{name}'
    if not os.path.exists(dest):
        print('downloading', name, '...')
        urllib.request.urlretrieve(url, dest)
    print(f'{name}: {os.path.getsize(dest) / 1e6:.0f} MB')

other_cells.h5ad: 59 MB
dcs.h5ad: 347 MB


## Step 1 — register the collection

One call references both files into a small repository. The expression matrices
(hundreds of MB) are *not* copied.

In [2]:
from virtual_h5ad import virtualize_matrix

virtualize_matrix('data/allen', 'repo/allen')

src_mb = sum(os.path.getsize(f'data/allen/{n}') for n in FILES) / 1e6
repo_kb = sum(os.path.getsize(os.path.join(r, f))
              for r, _, fs in os.walk('repo/allen') for f in fs) / 1e3
print(f'{src_mb:.0f} MB of source files  ->  {repo_kb:.0f} KB repository '
      f'({src_mb * 1000 / repo_kb:.0f}x smaller)')

/Users/ian/.local/share/uv/python/cpython-3.12.12-macos-aarch64-none/lib/python3.12/contextlib.py:137: UserWarning: zarr v3 autosharding will be the default in the next minor release.
  return next(self.gen)
/Users/ian/.local/share/uv/python/cpython-3.12.12-macos-aarch64-none/lib/python3.12/contextlib.py:137: UserWarning: zarr v3 autosharding will be the default in the next minor release.
  return next(self.gen)
/Users/ian/.local/share/uv/python/cpython-3.12.12-macos-aarch64-none/lib/python3.12/contextlib.py:137: UserWarning: zarr v3 autosharding will be the default in the next minor release.
  return next(self.gen)
/Users/ian/.local/share/uv/python/cpython-3.12.12-macos-aarch64-none/lib/python3.12/contextlib.py:137: UserWarning: zarr v3 autosharding will be the default in the next minor release.
  return next(self.gen)
/Users/ian/.local/share/uv/python/cpython-3.12.12-macos-aarch64-none/lib/python3.12/contextlib.py:137: UserWarning: zarr v3 autosharding will be the default in the next

406 MB of source files  ->  3508 KB repository (116x smaller)


/Users/ian/.local/share/uv/python/cpython-3.12.12-macos-aarch64-none/lib/python3.12/contextlib.py:137: UserWarning: zarr v3 autosharding will be the default in the next minor release.
  return next(self.gen)
/Users/ian/.local/share/uv/python/cpython-3.12.12-macos-aarch64-none/lib/python3.12/contextlib.py:137: UserWarning: zarr v3 autosharding will be the default in the next minor release.
  return next(self.gen)
/Users/ian/.local/share/uv/python/cpython-3.12.12-macos-aarch64-none/lib/python3.12/contextlib.py:137: UserWarning: zarr v3 autosharding will be the default in the next minor release.
  return next(self.gen)
/Users/ian/.local/share/uv/python/cpython-3.12.12-macos-aarch64-none/lib/python3.12/contextlib.py:137: UserWarning: zarr v3 autosharding will be the default in the next minor release.
  return next(self.gen)
/Users/ian/.local/share/uv/python/cpython-3.12.12-macos-aarch64-none/lib/python3.12/contextlib.py:137: UserWarning: zarr v3 autosharding will be the default in the next

## Step 2 — open the whole study as one AnnData

`open_anndata` returns a normal `AnnData` spanning both files. Opening is
near-instant because **no expression data is read yet** — `X` is a lazy view over
the original files. The rich CELLxGENE cell metadata (`obs`) is stitched together
so you can immediately work across the whole study.

In [3]:
import time
from virtual_h5ad import open_anndata

t = time.time()
adata = open_anndata('repo/allen')
print(f'opened {adata.shape[0]:,} cells x {adata.shape[1]:,} genes in {time.time() - t:.2f}s '
      f'(X is a lazy {type(adata.X).__name__} — no expression data read yet)')
adata

opened 34,224 cells x 32,357 genes in 0.13s (X is a lazy _CSRDataset — no expression data read yet)


AnnData object with n_obs × n_vars = 34224 × 32357
    obs: 'cohort.cohortGuid', 'sample.sampleKitGuid', 'specimen.specimenGuid', 'pipeline.fileGuid', 'subject.birthYear', 'subject.ageAtFirstDraw', 'subject.ageGroup', 'subject.race', 'subject.ethnicity', 'subject.cmv', 'subject.bmi', 'sample.visitName', 'sample.drawYear', 'sample.subjectAgeAtDraw', 'batch_id', 'pool_id', 'chip_id', 'well_id', 'original_barcodes', 'cell_name', 'n_reads', 'n_umis', 'n_genes', 'total_counts_mito', 'pct_counts_mito', 'doublet_score', 'AIFI_L1', 'AIFI_L2', 'AIFI_L3', 'disease_ontology_term_id', 'tissue_ontology_term_id', 'tissue_type', 'suspension_type', 'assay_ontology_term_id', 'sex_ontology_term_id', 'donor_id', 'self_reported_ethnicity_ontology_term_id', 'development_stage_ontology_term_id', 'cell_type_ontology_term_id', 'is_primary_data', 'cell_type', 'assay', 'disease', 'sex', 'tissue', 'self_reported_ethnicity', 'development_stage', 'observation_joinid'
    var: 'mito', 'n_cells_by_counts', 'mean_cou

In [4]:
# Real metadata, across both files — e.g. the cell-type composition:
adata.obs['cell_type'].value_counts().head(10)

cell_type
CD1c-positive myeloid dendritic cell     14235
platelet                                  7903
plasmacytoid dendritic cell               7587
erythrocyte                               1508
common myeloid progenitor                 1075
CD141-positive myeloid dendritic cell      943
Axl+ dendritic cell, human                 522
common lymphoid progenitor                 373
basophil mast progenitor cell               78
Name: count, dtype: int64

In [5]:
import glob, time
import anndata as ad

# What you read is exactly the original data — confirm bit-for-bit against the files.
row, identical = 0, True
for p in sorted(glob.glob('data/allen/*.h5ad')):
    src = ad.read_h5ad(p).X
    identical &= (adata[row:row + src.shape[0]].X != src).nnz == 0
    row += src.shape[0]
print('virtual X is byte-identical to the source files:', identical)

# ...and access is lazy: pull 500 cells on demand, straight from the originals.
t = time.time()
block = adata[1000:1500].X
print(f'read 500 cells in {time.time() - t:.3f}s  ->  {block.shape}, {block.nnz} nonzeros')

virtual X is byte-identical to the source files: True
read 500 cells in 0.058s  ->  (500, 32357), 1475215 nonzeros


## Step 3 — pull cells across the collection on demand

Select cells spanning both files; only those cells are read from disk.

In [6]:
import numpy as np

# 100 cells from the start (first file) and 100 from the end (second file)
picks = np.r_[0:100, adata.n_obs - 100:adata.n_obs]
subset = adata[picks].to_memory()
print(f'pulled {subset.n_obs} cells spanning both files -> {subset.shape}, '
      f'{subset.X.nnz} nonzero entries')

pulled 200 cells spanning both files -> (200, 32357), 336099 nonzero entries


## Scaling up

The same `virtualize_matrix('data/allen', 'repo/allen')` call works on all eight
atlas files (1.8M cells) — the repository stays tiny because it only stores
references, and `open_anndata` reads expression data lazily from the original
files. Requirements: the files share genes (`var`) and store `X` as sparse CSR
(CELLxGENE's standard layout).